In [1]:
# Ignore all other GPUs except this one
!export CUDA_VISIBLE_DEVICES=0

In [2]:
import numpy as np         
import matplotlib.pyplot as plt     
from matplotlib.animation import FuncAnimation          
import torch      
import torch.nn as nn    
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from torch.utils.data import TensorDataset, DataLoader        
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 
import time      
from scipy.ndimage import uniform_filter1d    
import pandas as pd
import pickle 
import os
from IPython.display import HTML

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
##Plasma Parmeters in normalized units##
n_e = 1.0  # Normalized electron density
T_e = 1.0  # Normalized electron temperature (k_B T_e / eV)
omega_p = 1.0  # Plasma frequency in natural units
v_th = 1.0  # Thermal velocity in natural units
lambda_D = 1.0  # Debye length in natural units
c=1

#normalized length and time grid#
Lx = 1
Ly = 1
nx = 200 #number of spatial poitns
dx = Lx/nx 
factor = 10 #how much time wave has to propagate 
x = np.linspace(0,Lx,nx)
dt = .01 # time steps in plasma periods omeg_p**-1
nt = 200 #number of time steps

#normalized wavenumber
kx=2 * np.pi/Lx
ky=2 * np.pi/Ly
k = np.sqrt(kx**2 + ky**2)

#Dispersion relation for langmuir waves#
omega = np.sqrt(omega_p**2 + 3*(k**2)*(v_th**2))
tau = factor / omega
E0 = 1.0 # arbitrary units for wave amplitude
B0 = E0 / c

In [5]:
def analytical_solution(x, y, t, E0):

    Ex = E0 * (k / k) * np.sin(k*x + k*y - omega*t)
    Ey = -1.0 * E0 * (k / k) * np.sin(k*x + k*y - omega*t)
    Bz = -1.0 * (E0 / c) * np.sin(k*x + k*y - omega*t)
    Bz_dot = (omega*E0 / c) * np.cos(k*x + k*y - omega*t)

    return Ex, Ey, Bz, Bz_dot

def generate_data(nx, ny, nt, Lx, Ly, tau, E0):

    x = np.linspace(0, Lx, nx)
    y = np.linspace(0, Ly, ny)
    t = np.linspace(0, tau, nt)

    dx = x[1] - x[0]
    dy = y[1] - y[0]
    dt = t[1] - t[0]

    x_arr, y_arr, t_arr =  np.meshgrid(x, y, t, indexing='ij')

    Ex, Ey, Bz, Bz_dot = analytical_solution(x_arr, y_arr, t_arr, E0)

    return x_arr.flatten(), y_arr.flatten(), t_arr.flatten(), \
           Ex.flatten(), Ey.flatten(), Bz.flatten(), Bz_dot.flatten(), \
           dx, dy, dt

def sparse_measurements(x, y, t, Ex, Ey, Bz, Bz_dot, num_samples):

    indices = np.random.choice(x.shape[0], num_samples, replace=False)

    return x[indices], y[indices], t[indices], Ex[indices], Ey[indices], Bz[indices], Bz_dot[indices]

def collocation_points(nx, ny, nt, Lx, Ly, tau):

    x = np.linspace(0, Lx, nx)
    y = np.linspace(0, Ly, ny)
    t = np.linspace(0, tau, nt)

    x_coll, y_coll, t_coll = np.meshgrid(x, y, t, indexing='ij')

    return x_coll.flatten(), y_coll.flatten(), t_coll.flatten()

In [6]:
Nx, Ny, Nt = 250, 250, 250

X_flat, Y_flat, T_flat, Ex_flat, Ey_flat, Bz_flat, Bz_dot_flat, dx, dy, dt = generate_data(Nx, Ny, Nt, Lx, Ly, tau, E0)
x_sparse, y_sparse, t_sparse, ex_sparse, ey_sparse, bz_sparse, bz_dot_sparse = sparse_measurements(X_flat, Y_flat, T_flat, \
                                                                                                   Ex_flat, Ey_flat, Bz_flat, Bz_dot_flat, \
                                                                                                   num_samples=10000)
x_coll, y_coll, t_coll = collocation_points(Nx, Ny, Nt, Lx, Ly, tau)

In [7]:
# #IC: Small perturbation in electrostatic potential 
# phi = np.sin(k*x)
# E = -np.gradient(phi, dx)
# E_new = np.zeros(nx)
# E_time = np.zeros((nt,nx))

# # Time loop
# for t in range(nt):
#     # Update electric field with normalized frequency
#     E_new = E * np.cos(omega * t * dt)
    
#     # Store electric field for animation
#     E_time[t, :] = E_new
    

# # Create an animation of the wave
# fig, ax = plt.subplots()
# line, = ax.plot(x, E_time[0, :])

# def update(frame):
#     line.set_ydata(E_time[frame, :])
#     return line,
# plt.title("1D Langmuir Wave")
# ani = FuncAnimation(fig, update, frames=nt, interval=50, blit=True)
# HTML(ani.to_jshtml())
